# Fine-tune Qwen2.5-7B for CV extraction (QLoRA) — Kaggle notebook

**What it does:** teaches Qwen2.5-7B-Instruct to fill our CV format (`CVExtraction`, DSO1) from about
1,000 CVs of the Resume Parser **train** split. The labels were written by an LLM (*silver*). Every CV of
our evaluation set (validation/test splits) is excluded. **Output:** a LoRA *adapter* (~160 MB) in
`/kaggle/working/adapter/`, then evaluated with `python scripts/kaggle_run.py all extraction-ft`.

**Settings (right panel)**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (not P100: no longer supported by PyTorch) |
| Internet | **On** |
| Persistence | *Files only* (optional: keeps `/kaggle/working` between interactive sessions) |

**Inputs (Add Input):** dataset `recruitment-ai-eval-v1` (our code; attached automatically by
`scripts/kaggle_run.py`). Optional: *Qwen2.5 7B Instruct* from Kaggle Models (saves the 15 GB download);
the output of an earlier run of this notebook (to resume it).

**Optional backup:** Add-ons → Secrets → `HF_TOKEN` (a Hugging Face *write* token), and set `HF_BACKUP_REPO`
below: every checkpoint is also copied to a private Hugging Face repository.

**Run it with _Save Version → Save & Run All (Commit)_.** It runs in the background and the browser can
be closed. An interactive session stops when the browser closes or stays idle.

**Interruptions:** every cell ends with a *saved and exported* line; a checkpoint is saved every
`SAVE_EVERY_MINUTES`; training stops by itself `RESERVE_MINUTES` before `MAX_SESSION_HOURS`, so the
outputs are always written. Running the notebook again (earlier output attached, or Hugging Face backup)
**resumes** from the last checkpoint.

**GPU hours (estimates, 2x T4):** setup ~15 min, training ~1.5-2 h per 1,000 CVs, final check ~10 min.

In [ ]:
# ════════════ 1. PARAMETERS — the only cell to edit ════════════
PACKAGE_ID = "local"        # filled in by scripts/kaggle_run.py ("local" = no dataset-version check)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # base model (Hugging Face id)
KAGGLE_MODEL_DIR = ""       # optional: folder of this model attached from Kaggle Models (also found automatically)

# Training data: Resume Parser train split (validation/test CVs excluded)
N_TRAIN = 1000              # training CVs: more = better, but ~1.5-2 GPU hours per 1,000 on 2x T4
N_HOLDOUT = 20              # train-split CVs kept aside for the final check
MIN_GROUNDEDNESS = 0.8      # skip CVs whose labels are not found in their text (LLM-made labels)
MAX_SEQ_LEN = 4096          # longer examples are skipped (T4 memory); lower it after an out-of-memory error

# QLoRA
EPOCHS = 1
LEARNING_RATE = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
BATCH_SIZE, GRAD_ACCUM = 1, 4  # effective batch = BATCH_SIZE x GRAD_ACCUM x number of GPUs
SEED = 42

# Kaggle safety
MAX_SESSION_HOURS = 7.0     # plan for a session that can break after 7 h
RESERVE_MINUTES = 40        # kept after training for the final check and the export
SAVE_EVERY_MINUTES = 15     # checkpoint frequency: an interruption loses at most this much training
SMOKE_TEST_CVS = 5          # held-out CVs extracted with the fine-tuned model at the end (0 = skip)
HF_BACKUP_REPO = ""         # optional: "<hf-user>/recruitment-qwen-cv-lora" (needs the Kaggle secret HF_TOKEN)
FORCE_RETRAIN = False       # True = train again even if a finished adapter is found

import json, pathlib, time

PARAMS = {k: globals()[k] for k in (
    "PACKAGE_ID", "MODEL_ID", "KAGGLE_MODEL_DIR", "N_TRAIN", "N_HOLDOUT", "MIN_GROUNDEDNESS", "MAX_SEQ_LEN",
    "EPOCHS", "LEARNING_RATE", "LORA_R", "LORA_ALPHA", "LORA_DROPOUT", "BATCH_SIZE", "GRAD_ACCUM", "SEED",
    "MAX_SESSION_HOURS", "RESERVE_MINUTES", "SAVE_EVERY_MINUTES", "SMOKE_TEST_CVS", "HF_BACKUP_REPO", "FORCE_RETRAIN")}
SESSION_START = time.time()
pathlib.Path("/kaggle/working/finetune").mkdir(parents=True, exist_ok=True)
pathlib.Path("/kaggle/working/finetune/params.json").write_text(json.dumps(PARAMS, indent=1))
print("✅ Cell 1/9 (parameters) — saved and exported: /kaggle/working/finetune/params.json")

In [ ]:
# ════════════ 2. Setup and checks (fails fast, before GPU time is wasted) ════════════
import gc, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

WORK = Path("/kaggle/working")
OUT, CKPT, ADAPTER, DATA = WORK / "finetune", WORK / "checkpoints", WORK / "adapter", WORK / "data"
for folder in (OUT, CKPT, DATA):
    folder.mkdir(parents=True, exist_ok=True)
STATE_FILE = OUT / "run_state.json"
os.environ["PYTHONUNBUFFERED"] = "1"        # live logs
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def minutes_left() -> float:
    """Minutes left before MAX_SESSION_HOURS (counted from the parameters cell)."""
    return MAX_SESSION_HOURS * 60 - (time.time() - SESSION_START) / 60


def state(**updates) -> dict:
    """Progress of the run, kept in /kaggle/working/finetune/run_state.json."""
    current = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    current.update(updates)
    STATE_FILE.write_text(json.dumps(current, indent=1, default=str))
    return current


def exported(cell: str, *paths) -> None:
    """End of a cell: check its files exist, record it, print the 'saved and exported' line."""
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise RuntimeError(f"Cell {cell}: expected outputs are missing: {missing}")
    state(**{f"cell_{cell.split('/')[0]}_done": time.strftime("%Y-%m-%d %H:%M:%S")})
    print(f"✅ Cell {cell} — saved and exported: {', '.join(str(p) for p in paths)} "
          f"| {minutes_left():.0f} min left in the session budget", flush=True)


def sh(cmd: str) -> int:
    """Run a shell command; its output is streamed into the cell."""
    print("+", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


import torch

gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", gpus or "none")
if not gpus:
    raise SystemExit("No GPU. Settings → Accelerator → 'GPU T4 x2', then run again (no GPU time was used).")
if any("P100" in g for g in gpus):
    raise SystemExit("P100 GPU: recent PyTorch and bitsandbytes no longer support it. Choose 'GPU T4 x2'.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except OSError:
    raise SystemExit("No internet. Settings → Internet → On (needs a phone-verified Kaggle account).")
disk = {p: round(shutil.disk_usage(p).free / 1e9) for p in ("/kaggle/working", "/root")}
print("free disk (GB):", disk)
if disk["/root"] < 25:
    print("⚠️ less than 25 GB free for the model cache: the 15 GB model download may fail")

try:  # optional Kaggle secret: Hugging Face backup and faster downloads
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception:
    print("no HF_TOKEN secret (fine unless HF_BACKUP_REPO is set)")
if HF_BACKUP_REPO and not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_BACKUP_REPO is set but the HF_TOKEN secret is missing (Add-ons → Secrets).")

# torch comes with Kaggle's image; only the training libraries are added
if sh("pip install -q -U 'transformers>=4.56' 'peft>=0.13' 'bitsandbytes>=0.45' 'accelerate>=1.0' "
      "pydantic-settings email-validator"):
    raise SystemExit("pip install failed (see above)")
import importlib.metadata as md

env = {"gpus": gpus, "python": sys.version.split()[0], "free_disk_gb": disk,
       **{p: md.version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")}}
(OUT / "environment.json").write_text(json.dumps(env, indent=1))
print(env)
exported("2/9 (setup)", OUT / "environment.json")

In [ ]:
# ════════════ 3. Project code (from the attached dataset) ════════════
import tarfile

PROJECT = Path("/tmp/project")
if not (PROJECT / "src").is_dir():
    archive = next(Path("/kaggle/input").rglob("project.tar.gz"), None)
    if archive:
        with tarfile.open(archive) as tar:
            tar.extractall(PROJECT.parent, filter="data")
    else:  # Kaggle unpacks the archives of datasets
        found = next((p for p in Path("/kaggle/input").rglob("project") if (p / "src").is_dir()), None)
        if found is None:
            raise SystemExit("Project code not found. Add Input → Your Work → Datasets → recruitment-ai-eval-v1.")
        shutil.copytree(found, PROJECT)
attached = (PROJECT / "PACKAGE_ID").read_text().strip() if (PROJECT / "PACKAGE_ID").exists() else "none"
if PACKAGE_ID != "local" and attached != PACKAGE_ID:
    raise SystemExit(f"Old dataset version attached ({attached}, expected {PACKAGE_ID}): Kaggle was still "
                     "processing the upload. Push the notebook again.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT / "src")  # for the training processes started by torchrun
(OUT / "project_version.txt").write_text(attached)
print("project code:", PROJECT, "| package", attached)
exported("3/9 (project code)", OUT / "project_version.txt")

In [ ]:
# ════════════ 4. Earlier progress (resume after an interruption) ════════════
import re

from transformers.trainer_utils import get_last_checkpoint


def _step(path: Path) -> int:
    match = re.search(r"checkpoint-(\d+)$", path.name)
    return int(match.group(1)) if match else -1


def copy_previous_data(previous_output: Path) -> None:
    """Reuse the exact training file of the interrupted run (same examples, same order)."""
    for name in ("train.jsonl", "holdout.jsonl"):
        if (previous_output / "data" / name).exists():
            shutil.copy(previous_output / "data" / name, DATA / name)
    if (previous_output / "finetune" / "data_report.json").exists():
        shutil.copy(previous_output / "finetune" / "data_report.json", OUT / "data_report.json")


def find_progress() -> str:
    """'adapter' (training finished), 'checkpoint' (resume) or 'none'; copies what it finds to /kaggle/working."""
    # 1. this session: a re-run after an error, or Kaggle's "Persistence: Files only"
    if (ADAPTER / "adapter_config.json").exists():
        return "adapter"
    if get_last_checkpoint(str(CKPT)):
        return "checkpoint"
    # 2. the output of an earlier run of this notebook, attached as an input
    inputs = Path("/kaggle/input")
    for config in inputs.rglob("adapter/adapter_config.json"):
        shutil.copytree(config.parent, ADAPTER, dirs_exist_ok=True)
        copy_previous_data(config.parent.parent)
        return "adapter"
    checkpoints = sorted((p for p in inputs.rglob("checkpoint-*") if p.is_dir()), key=_step)
    if checkpoints:
        shutil.copytree(checkpoints[-1], CKPT / checkpoints[-1].name, dirs_exist_ok=True)
        copy_previous_data(checkpoints[-1].parent.parent)
        return "checkpoint"
    # 3. the Hugging Face backup
    if HF_BACKUP_REPO:
        from huggingface_hub import snapshot_download
        try:
            backup = Path(snapshot_download(HF_BACKUP_REPO, allow_patterns=["adapter/*", "last-checkpoint/*"]))
        except Exception as exc:
            print("no Hugging Face backup yet:", type(exc).__name__)
            return "none"
        if (backup / "adapter" / "adapter_config.json").exists():
            shutil.copytree(backup / "adapter", ADAPTER, dirs_exist_ok=True)
            return "adapter"
        if (backup / "last-checkpoint" / "trainer_state.json").exists():
            step = json.loads((backup / "last-checkpoint" / "trainer_state.json").read_text())["global_step"]
            shutil.copytree(backup / "last-checkpoint", CKPT / f"checkpoint-{step}", dirs_exist_ok=True)
            return "checkpoint"
    return "none"


if FORCE_RETRAIN:
    shutil.rmtree(ADAPTER, ignore_errors=True)
    shutil.rmtree(CKPT, ignore_errors=True)
    CKPT.mkdir()
PROGRESS = "none" if FORCE_RETRAIN else find_progress()
print({"adapter": "a finished adapter was found: training will be skipped (FORCE_RETRAIN = True to train again)",
       "checkpoint": f"resuming from {get_last_checkpoint(str(CKPT))}",
       "none": "no earlier progress: a new training run starts"}[PROGRESS])
state(progress_found=PROGRESS)
exported("4/9 (resume check)", STATE_FILE)

In [ ]:
# ════════════ 5. Training data (CPU only, ~2 min) ════════════
from recruitment_ai.data.resume_parser import iter_split
from recruitment_ai.training.data import build_training_set

TRAIN_FILE, HOLDOUT_FILE, DATA_REPORT = DATA / "train.jsonl", DATA / "holdout.jsonl", OUT / "data_report.json"
data_params = {k: PARAMS[k] for k in ("N_TRAIN", "N_HOLDOUT", "MIN_GROUNDEDNESS", "SEED")}
previous = json.loads(DATA_REPORT.read_text()) if DATA_REPORT.exists() else {}
if TRAIN_FILE.exists() and HOLDOUT_FILE.exists() and previous.get("params") == data_params:
    report = previous
    print("reusing the training data of the earlier run (same examples, same order)")
else:
    if PROGRESS == "checkpoint":
        print("⚠️ the data parameters changed since the checkpoint: set FORCE_RETRAIN = True for a clean run")
    from huggingface_hub import hf_hub_download
    split_file = {s: Path(hf_hub_download("md-zain/resume-parser-dataset", f"curated/{s}.jsonl",
                                          repo_type="dataset")) for s in ("train", "validation", "test")}
    eval_texts = [r["resume_text"] for s in ("validation", "test") for r in iter_split(split_file[s])]
    train_rows, holdout_rows, report = build_training_set(
        iter_split(split_file["train"]), eval_texts, N_TRAIN, N_HOLDOUT, seed=SEED, min_groundedness=MIN_GROUNDEDNESS)
    for path, rows in ((TRAIN_FILE, train_rows), (HOLDOUT_FILE, holdout_rows)):
        path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    report["params"] = data_params
    DATA_REPORT.write_text(json.dumps(report, indent=1))
print({k: v for k, v in report.items() if k != "train_categories"})
if report["train"] < N_TRAIN:
    print(f"⚠️ only {report['train']} usable training CVs (asked for {N_TRAIN})")
exported("5/9 (training data)", TRAIN_FILE, HOLDOUT_FILE, DATA_REPORT)

In [ ]:
# ════════════ 6. Base model weights (session cache only) ════════════
def attached_model() -> str | None:
    """A copy of MODEL_ID attached from Kaggle Models, if any (no download needed)."""
    if KAGGLE_MODEL_DIR:
        return KAGGLE_MODEL_DIR
    size = MODEL_ID.split("/")[-1].lower().replace("qwen2.5-", "")  # e.g. "7b-instruct"
    for config in Path("/kaggle/input").rglob("config.json"):
        folder = str(config.parent).lower()
        if (size in folder and "qwen" in folder and (config.parent / "tokenizer_config.json").exists()
                and json.loads(config.read_text()).get("model_type") == "qwen2"):
            return str(config.parent)
    return None


MODEL_PATH = attached_model()
if MODEL_PATH:
    print("using the model attached from Kaggle Models:", MODEL_PATH)
else:
    from huggingface_hub import snapshot_download
    print(f"downloading {MODEL_ID} from Hugging Face (~15 GB, a few minutes)...", flush=True)
    MODEL_PATH = snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.safetensors", "*.txt"])
state(model_path=MODEL_PATH)
# The weights stay in the session cache: 15 GB would not fit Kaggle's 20 GB output limit with the
# rest. Only their location is exported; a new session downloads them again.
exported("6/9 (base model)", STATE_FILE)

In [ ]:
# ════════════ 7. QLoRA training on every GPU (checkpoint every SAVE_EVERY_MINUTES) ════════════
TRAIN_STATE = OUT / "training_state.json"
if PROGRESS == "adapter":
    print("a finished adapter was found: training skipped")
    state(training="complete")
else:
    minutes_for_training = minutes_left() - RESERVE_MINUTES
    if minutes_for_training < 20:
        print(f"only {minutes_for_training:.0f} min left for training in this session: run again, it resumes")
        state(training="paused")
    else:
        config = {
            "model_path": MODEL_PATH, "train_file": str(TRAIN_FILE), "output_dir": str(CKPT),
            "adapter_dir": str(ADAPTER), "state_file": str(TRAIN_STATE), "max_seq_len": MAX_SEQ_LEN,
            "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "seed": SEED,
            "stop_at": time.time() + minutes_for_training * 60, "save_every_minutes": SAVE_EVERY_MINUTES,
            "hub_backup_repo": HF_BACKUP_REPO or None,
        }
        config_file = OUT / "train_config.json"
        config_file.write_text(json.dumps(config, indent=1))
        n_gpus = torch.cuda.device_count()
        print(f"training on {n_gpus} GPU(s), at most {minutes_for_training:.0f} min in this session", flush=True)
        code = sh(f"torchrun --standalone --nproc_per_node {n_gpus} scripts/finetune_qlora.py --config {config_file}")
        if code and n_gpus > 1 and not get_last_checkpoint(str(CKPT)):
            # failed before the first checkpoint: retry with a single process (no multi-GPU synchronisation)
            print("⚠️ multi-GPU training failed early: retrying on one GPU", flush=True)
            config["stop_at"] = time.time() + (minutes_left() - RESERVE_MINUTES) * 60
            config_file.write_text(json.dumps(config, indent=1))
            code = sh(f"torchrun --standalone --nproc_per_node 1 scripts/finetune_qlora.py --config {config_file}")
        status = json.loads(TRAIN_STATE.read_text())["status"] if code == 0 and TRAIN_STATE.exists() else "failed"
        state(training=status, training_exit_code=code)
        print({"complete": "✅ training complete: adapter in /kaggle/working/adapter",
               "paused": "⏸ session budget reached: the last checkpoint is saved; run again to resume",
               "failed": "❌ training failed (see the log above); the next cells still save what exists"}[status])
torch.cuda.empty_cache()
exported("7/9 (training)", STATE_FILE, CKPT)

In [ ]:
# ════════════ 8. Quick check: extract held-out CVs with the fine-tuned model ════════════
SMOKE_FILE = OUT / "smoke_test.json"
last_checkpoint = get_last_checkpoint(str(CKPT)) if CKPT.exists() else None
adapter = str(ADAPTER) if (ADAPTER / "adapter_config.json").exists() else last_checkpoint
if not SMOKE_TEST_CVS or adapter is None:
    result = {"skipped": "SMOKE_TEST_CVS = 0" if adapter else "no adapter yet"}
elif minutes_left() < 15:
    result = {"skipped": "not enough session time left"}
else:
    os.environ.update({"LLM_DEVICE": "cuda", "LLM_DTYPE": "float16"})
    from recruitment_ai.config.settings import get_settings
    get_settings.cache_clear()
    from recruitment_ai.evaluation.extraction_eval import score_profile, summarize
    from recruitment_ai.extraction.extractor import CVExtractor
    from recruitment_ai.llm.huggingface_adapter import HuggingFaceAdapter

    holdout = [json.loads(line) for line in HOLDOUT_FILE.read_text(encoding="utf-8").splitlines() if line]
    extractor = CVExtractor(llm=HuggingFaceAdapter(model=MODEL_PATH, adapter=adapter))
    rows = []
    for example in holdout[:SMOKE_TEST_CVS]:
        try:
            profile, stats = extractor.extract_text(example["text"], example["id"])
            prediction, error = profile.model_dump(mode="json", exclude={"created_at"}), None
        except Exception as exc:  # one bad answer must not stop the export
            prediction, stats, error = {}, {}, f"{type(exc).__name__}: {exc}"[:300]
        rows.append({"id": example["id"], "error": error, "seconds": stats.get("seconds"),
                     "scores": score_profile(example["reference"], prediction, extractor.normalizer)})
        print(f"  {example['id']}: {'ERROR ' + error if error else str(stats.get('seconds')) + ' s'}", flush=True)
    overall = summarize([r["scores"] for r in rows])
    result = {"adapter": adapter, "cvs": len(rows), "errors": sum(bool(r["error"]) for r in rows),
              "experience_f1": overall["experience"]["f1"], "education_f1": overall["education"]["f1"],
              "skills_f1": overall["skills"]["f1"], "overall_f1": overall["overall_f1"], "per_cv": rows}
    del extractor
    gc.collect()
    torch.cuda.empty_cache()
SMOKE_FILE.write_text(json.dumps(result, indent=1, default=str))
print({k: v for k, v in result.items() if k != "per_cv"})
exported("8/9 (quick check)", SMOKE_FILE)

In [ ]:
# ════════════ 9. Export: report, model card, output size ════════════
training = json.loads(TRAIN_STATE.read_text()) if TRAIN_STATE.exists() else {}
status = "complete" if (ADAPTER / "adapter_config.json").exists() else state().get("training", "not started")
if status == "complete" and CKPT.exists():
    shutil.rmtree(CKPT)  # the adapter is all that is needed; checkpoints only serve to resume
    CKPT.mkdir()
losses = training.get("losses", [])
check = globals().get("result", {})
lines = [
    "# QLoRA fine-tuning report", "",
    f"- Status: **{status}**",
    f"- Base model: `{MODEL_ID}`",
    f"- Training CVs: {report.get('train')} from the Resume Parser train split (silver labels; evaluation CVs excluded)",
    f"- Steps: {training.get('global_step', '?')}/{training.get('max_steps', '?')} on "
    f"{training.get('gpu_processes', '?')} GPU(s)",
    f"- Training loss: {losses[0]['loss']:.4f} at the start → {losses[-1]['loss']:.4f} at the end" if losses
    else "- Training loss: not available in this session",
    f"- Quick check on {check['cvs']} held-out CVs: overall F1 {check['overall_f1']}" if "overall_f1" in check
    else f"- Quick check: {check.get('skipped', 'not run')}",
    "", "Details: `params.json`, `data_report.json`, `training_state.json`, `smoke_test.json`.", "",
    "Next: " + {"complete": "evaluate on the frozen evaluation set: `python scripts/kaggle_run.py all extraction-ft`.",
                "paused": "run this notebook again (Save & Run All) with this version's output attached as an "
                          "input, or with HF_BACKUP_REPO set: it resumes from the last checkpoint."}
    .get(status, "read the log of cell 7, fix the cause, run again."),
]
(OUT / "report.md").write_text("\n".join(lines) + "\n")
if status == "complete":
    (ADAPTER / "README.md").write_text(
        f"# {MODEL_ID} + LoRA adapter for CV extraction (recruitment project, Step 9)\n\n"
        f"QLoRA (4-bit NF4, r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, all attention and MLP "
        f"projections), {EPOCHS} epoch(s) on {report.get('train')} CVs of md-zain/resume-parser-dataset (train "
        "split; labels written by an LLM, i.e. silver; validation/test CVs excluded).\n\n"
        "Use it with the project's prompt (recruitment_ai.extraction.extractor): `LLM_ADAPTER=<this folder>` or "
        "`python scripts/eval_extraction.py --adapter <this folder>`.\n")
sizes = {p.name: round(sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6, 1)
         for p in WORK.iterdir() if p.is_dir()}
print("output sizes (MB):", sizes)
if sum(sizes.values()) > 19_000:
    print("⚠️ close to Kaggle's 20 GB output limit")
print("\n".join(lines))
exported("9/9 (export)", OUT / "report.md")
if status not in ("complete", "paused"):
    raise SystemExit(f"Training {status}: everything that exists is saved; see the log of cell 7.")